# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

I made a 90 days time window selected from latest 3 months from date 2026-04-01 to 2026-06-30.<br><br>
**One row = One client's one reported content page needed refresh or not.**<br><br> Trained on 60day window (2026-04-01 to 2026-05-31), tested on June (month=06) 30day window (2026-06-01 to 2026-06-30).

<h2>1.1: Import libraries and data from hugging face</h2>

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h2>1.2: Checking start date and end date of data.</h2>

In [7]:
date_range = con.sql(f"""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        MAX(report_date) - MIN(report_date) as duration
    FROM {TABLES['fact_daily']}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│ start_date │  end_date  │ duration │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │      519 │
└────────────┴────────────┴──────────┘



<h2>1.3: Get 3 months latest window</h2>

In [8]:
clients_last_3m = con.sql(f"""
    SELECT report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-06-30'
""")

In [9]:
print(f"Rows in last 3 months: {len(clients_last_3m):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in last 3 months: 33,806,178


<h2>1.4: 90 day window start and end dates/ Time window</h2>

In [10]:
con.sql("""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM clients_last_3m
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│ start_date │  end_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-04-01 │ 2026-06-30 │
└────────────┴────────────┘



<h2>1.5: One unit of analysis</h2>

In [11]:
con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
    LIMIT 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

<h1>Features:</h1>
       'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
       'sessions_organic', 'sessions_direct', 'sessions_referral',
       'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta',
       'ai_other', 'scroll_events', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used',
       'model_used',
       ''

<h1>Proxy label: 'decline_score'</h1>

<h1>Context:</h1>
'report_date', 'client_hash_id', 'content_hash_id', 'url_hash_id', 'client_has_gsc', 'keyword_hash_id', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'content_created_date', 'content_updated_date', 'last_optimized_date','keyword_created_date', 'is_published', 'is_deleted',

<h1>Excluded:</h1>
1. month - report_date already shows month so no need<br>
2. optimization_eligible_date - leakage for output refresh_needed so dropped<br>
3. char_count - Not matter for declining signal<br>
4. word_count - Not matter for declining signal.<br>
5. category_count - Not matter for declining signal.<br>
6. keyword_char_count - Not matter for declining signal<br>
7. keyword_token_count - Not matter for declining signal.<br>
8. url_char_count - Not matter for declining signal<br>

<h2>2.1: 'fact_daily' combined with 'dim_c' training window</h2>

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
training_window = con.sql(f"""
    SELECT
        f.*,
        d.keyword_char_count,
        d.keyword_token_count,
        d.content_created_date,
        d.content_updated_date,
        d.content_type,
        d.search_volume,
        d.competition,
        d.competition_level,
        d.cpc,
        d.main_intent,
        d.backlinks,
        d.category_count,
        d.keyword_created_date,
        d.provider_used,
        d.model_used,
        d.char_count,
        d.word_count,
        d.last_optimized_date,
        d.optimization_eligible_date,
        d.is_published,
        d.is_deleted
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id
      AND f.content_hash_id = d.content_hash_id
    WHERE f.report_date >= '2026-04-01' AND f.report_date <= '2026-05-31'
""")


<h2>2.1.1: Total columns</h2>

In [13]:
training_window.limit(0).show()

┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┬────────────────────┬─────────────────────┬──────────────────────┬──────────────────────┬──────────────┬───────────────┬─────────────┬───────────────────┬────────┬─────────────┬───────────┬────────────────┬──────────────────────┬───────────────┬────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc

<h2>2.1.2: Columns as 'Features'</h2>

<h2>Features</h2>

In [14]:
features = training_window.select("""
    gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position,
    ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,
    ga4_total_engagement_sec, sessions_organic, sessions_direct,
    sessions_referral, sessions_social, sessions_paid, sessions_ai,
    ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude,
    ai_meta, ai_other, scroll_events, content_type, search_volume,
    competition, competition_level, cpc, main_intent, backlinks,
    provider_used, model_used
""")

print("=== Features Schema ===")
features.limit(0).show()

=== Features Schema ===
┌─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬──────────────┬───────────────┬─────────────┬───────────────────┬────────┬─────────────┬───────────┬───────────────┬────────────┐
│ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_events │ content_type │ search_volume │ competition │ competition_level │  cpc   │ main_intent │ backlinks │ pro

<h2>2.1.3: Columns as 'Context'</h2>

<h2>Context</h2>

In [15]:
context = training_window.select("""
    report_date, client_hash_id, content_hash_id, client_has_gsc,
    client_has_ga4, gsc_data_available, ga4_data_available,
    content_created_date, content_updated_date, keyword_created_date,
    last_optimized_date, is_published, is_deleted
""")

print("=== Context Schema ===")
context.limit(0).show()

=== Context Schema ===
┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬──────────────────────┬──────────────────────┬──────────────────────┬─────────────────────┬──────────────┬────────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ content_created_date │ content_updated_date │ keyword_created_date │ last_optimized_date │ is_published │ is_deleted │
│    date     │    varchar     │     varchar     │    boolean     │    boolean     │      boolean       │      boolean       │         date         │         date         │         date         │        date         │   boolean    │  boolean   │
├─────────────┴────────────────┴─────────────────┴────────────────┴────────────────┴────────────────────┴────────────────────┴──────────────────────┴──────────────────────┴──────────────────────┴─────────────────────┴──────────────┴───────

<h2>2.1.4: Columns as 'Excluded'</h2>

<h2>Excluded</h2>

In [16]:
excluded = training_window.select("""
    month, keyword_char_count, keyword_token_count, category_count,
    char_count, word_count, optimization_eligible_date
""")
print("=== Excluded Schema ===")
excluded.limit(0).show()

=== Excluded Schema ===
┌─────────┬────────────────────┬─────────────────────┬────────────────┬────────────┬────────────┬────────────────────────────┐
│  month  │ keyword_char_count │ keyword_token_count │ category_count │ char_count │ word_count │ optimization_eligible_date │
│ varchar │       int64        │        int64        │     int64      │   int64    │   int64    │            date            │
├─────────┴────────────────────┴─────────────────────┴────────────────┴────────────┴────────────┴────────────────────────────┤
│                                                           0 rows                                                           │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

<h2>3.1: Time Window</h2>

In [17]:
con.sql("""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM clients_last_3m
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│ start_date │  end_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-04-01 │ 2026-06-30 │
└────────────┴────────────┘



<h2>3.2: Total grain count</h2>

In [18]:
print(f"Rows in first month: {len(training_window):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in first month: 22,112,106


<h2>3.3: No duplication of any row</h2>

In [19]:
# Verification: Check for duplicates in training_window for the primary keys
duplicates = training_window.aggregate(
    "report_date, client_hash_id, content_hash_id, count(*) AS counts",
    "report_date, client_hash_id, content_hash_id"
).filter("counts > 1")

if duplicates.fetchone() is None:
    print("Success: No duplicate rows found for the grain (date, client, content).")
else:
    print(f"Warning: Found duplicates!")
    duplicates.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success: No duplicate rows found for the grain (date, client, content).


<h2>3.4: Checking missing values</h2>

Missing/Null values does not mean the data has ambiguity, they are just missing because in real world that value just does not come up.<br>
We cannot remove rows having very large null count as it can be a useful real signal for the label decision.

In [20]:
cols = con.sql("SELECT * FROM training_window LIMIT 0").columns

for col in cols:
    result = con.sql(f"""
        SELECT COUNT(*) FILTER (WHERE {col} IS NULL) AS null_count
        FROM training_window
    """).fetchone()[0]
    print(f"{col}: {result} nulls")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

report_date: 0 nulls


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

client_hash_id: 0 nulls


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

optimization_eligible_date: 19392209 nulls


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_published: 0 nulls


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_deleted: 0 nulls


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced history- gsc and ga4 start dates differ for pages, this means the data can never give you a fair comparison across all 104 clients equally, some clients simply have much more history than others.

In [21]:
print("=== Proof: GA4 vs GSC Imbalance & History Differences ===")
con.sql("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available = true AND gsc_data_available = true) as both_available,
        COUNT(*) FILTER (WHERE gsc_data_available = true AND (ga4_data_available = false OR ga4_data_available IS NULL)) as gsc_only,
        COUNT(DISTINCT client_hash_id) as total_clients
    FROM training_window
""").show()

=== Proof: GA4 vs GSC Imbalance & History Differences ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬──────────┬───────────────┐
│ total_rows │ both_available │ gsc_only │ total_clients │
│   int64    │     int64      │  int64   │     int64     │
├────────────┼────────────────┼──────────┼───────────────┤
│   22112106 │        1129081 │  7145401 │            66 │
└────────────┴────────────────┴──────────┴───────────────┘



In [22]:
print("=== Proof: History Variance per Client ===")
con.sql("""
    SELECT
        client_hash_id,
        MIN(report_date) as first_seen,
        MAX(report_date) as last_seen,
        COUNT(DISTINCT report_date) as active_days
    FROM training_window
    GROUP BY client_hash_id
    ORDER BY active_days ASC
    LIMIT 10
""").show()

=== Proof: History Variance per Client ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────┬────────────┬─────────────┐
│     client_hash_id      │ first_seen │ last_seen  │ active_days │
│         varchar         │    date    │    date    │    int64    │
├─────────────────────────┼────────────┼────────────┼─────────────┤
│ client_aef6ffea193da149 │ 2026-05-28 │ 2026-05-31 │           4 │
│ client_04660893ae39614a │ 2026-05-22 │ 2026-05-31 │          10 │
│ client_a22068e339bf95f5 │ 2026-05-22 │ 2026-05-31 │          10 │
│ client_c353557474475e51 │ 2026-04-30 │ 2026-05-31 │          22 │
│ client_1a8bf67cad4ee525 │ 2026-05-09 │ 2026-05-31 │          23 │
│ client_c7c2962f1c9c3089 │ 2026-05-09 │ 2026-05-31 │          23 │
│ client_7de9989c909e91a5 │ 2026-04-20 │ 2026-05-31 │          42 │
│ client_9c26c096d6e57253 │ 2026-04-16 │ 2026-05-31 │          46 │
│ client_0b245132bb722950 │ 2026-04-06 │ 2026-05-31 │          56 │
│ client_06d356715a8ff3b6 │ 2026-04-02 │ 2026-05-31 │          60 │
├─────────────────────────┴────────────┴────────

2. Likewise, most of gsc_avg_position(=13837635) missing also


In [23]:
print("\n=== Proof: GSC Average Position Missingness ===")
con.sql("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) as missing_avg_pos,
        (COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) * 100.0 / COUNT(*)) as pct_missing
    FROM training_window
""").show()


=== Proof: GSC Average Position Missingness ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────┬───────────────────┐
│ total_rows │ missing_avg_pos │    pct_missing    │
│   int64    │      int64      │      double       │
├────────────┼─────────────────┼───────────────────┤
│   22112106 │        13837635 │ 62.57945308330197 │
└────────────┴─────────────────┴───────────────────┘



3. Even with perfect signals, the signals are observational and can never tell why such page behaves like this.


4. Window does not overlap, we can never know any signal leading to the outcome from 1 month window.


In [24]:
print("=== Proof: Window Non-Overlap (Training vs Test) ===")
con.sql("""
    SELECT
        month,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        COUNT(*) as row_count
    FROM training_window
    GROUP BY month
    ORDER BY month
""").show()

=== Proof: Window Non-Overlap (Training vs Test) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┬───────────┐
│  month  │ start_date │  end_date  │ row_count │
│ varchar │    date    │    date    │   int64   │
├─────────┼────────────┼────────────┼───────────┤
│ 2026-04 │ 2026-04-01 │ 2026-04-30 │  10424730 │
│ 2026-05 │ 2026-05-01 │ 2026-05-31 │  11687376 │
└─────────┴────────────┴────────────┴───────────┘



## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.